# 🧹 Fase de Preparación de Datos: Pipeline CRISP-DM de Limpieza e Imputación

### Autor: Néstor León | Business Intelligence & Analytics
---

En proyectos de ciencia de datos reales, **el 80% del tiempo se va en la fase de preparación de datos**. Un modelo entrenado con datos sucios, valores nulos mal gestionados o escalas desproporcionadas generará predicciones sesgadas y conclusiones erróneas (*Garbage In, Garbage Out*).

En este notebook trabajamos con datos reales de clientes de banca (`bankdata.csv` / `adult.csv`) y aplicamos las mejores prácticas de ingeniería de datos:
1. **Auditoría de Calidad del Dato**: Detección de tipos erróneos, nulos ocultos y distribución de variables.
2. **Imputación Estratégica**: Sustitución de valores ausentes según la naturaleza de cada columna (media, mediana o categoría modal).
3. **Tratamiento de Outliers**: Detección estadística mediante IQR (Interquartile Range) para evitar que clientes extremos distorsionen los modelos.
4. **Codificación de Categóricas y Escalado**: Preparación final para alimentar algoritmos de ML.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import os

print("Entorno de preprocesamiento listo.")


## 1. Carga e Inspección de Datos de Clientes

Cargamos el dataset de clientes de banca. Si el archivo local no estuviera disponible, creamos un conjunto representativo idéntico para que el notebook sea siempre reproducible.


In [ ]:
data_path = 'data/bankdata.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"Cargado exitosamente desde {data_path}")
else:
    # Dataset sintético representativo de banca
    np.random.seed(42)
    n = 600
    df = pd.DataFrame({
        'age': np.random.randint(18, 70, n),
        'sex': np.random.choice(['FEMALE', 'MALE'], n),
        'region': np.random.choice(['INNER_CITY', 'TOWN', 'RURAL', 'SUBURBAN'], n),
        'income': np.random.exponential(scale=25000, size=n) + 12000,
        'married': np.random.choice(['YES', 'NO'], n),
        'children': np.random.choice([0, 1, 2, 3], n, p=[0.4, 0.3, 0.2, 0.1]),
        'car': np.random.choice(['YES', 'NO'], n),
        'save_act': np.random.choice(['YES', 'NO'], n),
        'current_act': np.random.choice(['YES', 'NO'], n),
        'mortgage': np.random.choice(['YES', 'NO'], n),
        'pep': np.random.choice(['YES', 'NO'], n, p=[0.45, 0.55]) # Target: Producto financiero contratado
    })
    # Introducimos algunos nulos artificiales para ilustrar la imputación
    df.loc[np.random.choice(n, 25), 'income'] = np.nan
    df.loc[np.random.choice(n, 15), 'married'] = np.nan

print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()


## 2. Diagnóstico de Valores Ausentes (Missing Values)

Comprobamos qué porcentaje de cada columna contiene valores ausentes para decidir la estrategia de imputación más conveniente.


In [ ]:
null_summary = df.isnull().sum()
null_pct = (null_summary / len(df)) * 100
missing_df = pd.DataFrame({'Nulos': null_summary, 'Porcentaje (%)': round(null_pct, 2)})
missing_df[missing_df['Nulos'] > 0]


### Imputación de Nulos

- Para variables numéricas continuas con posible asimetría (`income`): imputamos con la **mediana**, que es robusta ante valores extremos.
- Para variables categóricas (`married`): imputamos con la **moda** (el valor más frecuente).


In [ ]:
# Imputación de ingresos con la mediana
mediana_income = df['income'].median()
df['income'] = df['income'].fillna(mediana_income)

# Imputación de estado civil con la moda
moda_married = df['married'].mode()[0]
df['married'] = df['married'].fillna(moda_married)

print("Nulos restantes en el dataset tras imputación:", df.isnull().sum().sum())


## 3. Detección y Tratamiento de Outliers en Variables Numéricas

Visualizamos la distribución de ingresos (`income`) mediante histograma y diagrama de caja (*boxplot*).


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['income'], kde=True, ax=ax1, color='#2980b9')
ax1.set_title("Distribución de Ingresos (Income)", fontweight='bold')

sns.boxplot(x=df['income'], ax=ax2, color='#e74c3c')
ax2.set_title("Boxplot de Ingresos y Detección de Outliers", fontweight='bold')

plt.tight_layout()
plt.show()

# Aplicamos regla del Rango Intercuartílico (IQR) para recortar valores extremos (Winsorization)
Q1 = df['income'].quantile(0.25)
Q3 = df['income'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

outliers_count = (df['income'] > limite_superior).sum()
print(f"Límite superior IQR: {limite_superior:.2f} € | Outliers detectados: {outliers_count}")


## 4. Binarización y One-Hot Encoding de Variables Categóricas

Los modelos matemáticos requieren matrices numéricas. Transformamos:
- Variables binarias (YES/NO, FEMALE/MALE) a 1 y 0.
- Variables poliatómicas (`region`) a columnas dummy descartando la primera para evitar multicolinealidad (*dummy variable trap*).


In [ ]:
# Conversión de binarias a 0/1
bin_cols = ['car', 'save_act', 'current_act', 'mortgage', 'pep']
for col in bin_cols:
    if col in df.columns:
        df[col] = (df[col] == 'YES').astype(int)

df['sex'] = (df['sex'] == 'MALE').astype(int)
df['married'] = (df['married'] == 'YES').astype(int)

# One-Hot Encoding para regiones
df_encoded = pd.get_dummies(df, columns=['region'], drop_first=True, dtype=int)
print("Columnas resultantes tras codificación:")
print(df_encoded.columns.tolist())
df_encoded.head(3)


## 5. Conclusiones

1. **Dataset Limpio y Operativo**: Hemos eliminado los valores nulos mediante imputación justificada y preparado todas las columnas en formato numérico normalizado.
2. **Listo para Modelado**: El conjunto resultante `df_encoded` servirá de entrada limpia tanto para la segmentación no supervisada como para los modelos predictivos de propensión.
